# Grid Search over the Leader Policy

In [ ]:
import os
import sys
import time
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd
import torch
from torch import nn
from omegaconf import OmegaConf
from garage.torch import set_gpu_mode
from garage.torch.q_functions import DiscreteMLPQFunction
from garage.experiment.deterministic import set_seed, get_seed

from src.experiment import (
    Trainer, 
    wrap_experiment, 
    set_seed as custom_set_seed,
    kwargs_from_cfg, get_algo, 
    args_for_experiments
)
from src.policies import CategoricalMLPPolicy, JointPolicy
from src.envs import GymEnv
from src.replay_buffer import GammaReplayBuffer
from src.follower import FollowerWrapper, SoftQIteration
from src.sampler import LocalSampler, VecWorker

In [ ]:
# ===== Notebook parameters (equivalent to argparse args) =====
CONFIG_PATH = '../config/DiscreteToy4_R1-v0/config_bchg.yaml'
GRID_START = 0.0
GRID_STOP = 1.0
GRID_STEP = 0.1
SEEDS = [0,1,2,3,4]
OUTPUT_DIR = '../data/local/experiment/grid_search_DiscreteToy4_R1-v0'

In [ ]:
# Load base config
config = OmegaConf.load(CONFIG_PATH)
print(f'Loaded config: {CONFIG_PATH}')

print('Grid search parameters:')
print(f'  grid_start: {GRID_START}')
print(f'  grid_stop: {GRID_STOP}')
print(f'  grid_step: {GRID_STEP}')
print(f'  seeds: {SEEDS}')

if torch.cuda.is_available():
    set_gpu_mode(True)
else:
    set_gpu_mode(False)

if config.get('set_detect_anomaly', False):
    torch.autograd.set_detect_anomaly(True)

grid_values = np.arange(GRID_START, GRID_STOP + 1e-6, GRID_STEP).round(2)
print(f'\nFixed action grid values: {grid_values}')

In [ ]:
class FixedActionCategoricalPolicy(CategoricalMLPPolicy):
    """Categorical policy that always selects the same discrete action."""

    def __init__(self,
                 env_spec,
                 fixed_action=1,
                 hidden_sizes=(64, 64),
                 hidden_nonlinearity=nn.ReLU,
                 output_nonlinearity=None,
                 name='FixedActionCategoricalPolicy'):
        if not hasattr(env_spec.action_space, 'n'):
            raise ValueError('FixedActionCategoricalPolicy requires a discrete action space.')
        self._is_prob = False
        if isinstance(fixed_action, float):
            if env_spec.action_space.n != 2:
                raise ValueError('Probability-style fixed_action requires binary action space (n=2).')
            if fixed_action < 0.0 or fixed_action > 1.0:
                raise ValueError(f'fixed_action probability must be in [0.0, 1.0], got {fixed_action}.')
            self._fixed_action_prob = float(fixed_action)
            self._is_prob = True
        else:
            if fixed_action < 0 or fixed_action >= env_spec.action_space.n:
                raise ValueError(
                    f'fixed_action must be in [0, {env_spec.action_space.n - 1}], got {fixed_action}.')
            self._fixed_action = int(fixed_action)
        super().__init__(
            env_spec=env_spec,
            hidden_sizes=hidden_sizes,
            hidden_nonlinearity=hidden_nonlinearity,
            output_nonlinearity=output_nonlinearity,
            name=name,
        )

    def forward(self, observations):
        batch_size = observations.shape[0]
        if self._is_prob:
            p = self._fixed_action_prob
            probs = torch.empty(
                batch_size,
                self.action_dim,
                device=observations.device,
                dtype=torch.float32,
            )
            probs[:, 0] = 1.0 - p
            probs[:, 1] = p
            dist = torch.distributions.Categorical(probs=probs)
            mode_val = 1 if p >= 0.5 else 0
            mode = torch.full(
                (batch_size,),
                mode_val,
                device=observations.device,
                dtype=torch.long,
            )
            return dist, {'mode': mode, 'probs': probs}
        else:
            probs = torch.zeros(
                batch_size,
                self.action_dim,
                device=observations.device,
                dtype=torch.float32,
            )
            probs[:, self._fixed_action] = 1.0
            dist = torch.distributions.Categorical(probs=probs)
            mode = torch.full(
                (batch_size,),
                self._fixed_action,
                device=observations.device,
                dtype=torch.long,
            )
            return dist, {'mode': mode, 'probs': probs}

class FixedLeaderSoftQIteration(SoftQIteration):
    """SoftQIteration variant that handles a fixed leader policy cleanly."""

    def train_once(self, trainer):
        with torch.no_grad():
            Q_prev = self.Q.clone()
            Q_new = torch.zeros_like(self.Q)
            leader_policy = trainer.leader.policy
            if self.num_leader_actions != 2:
                raise ValueError(
                    'FixedLeaderSoftQIteration expects a binary leader action space (n=2).'
                )
            if getattr(leader_policy, '_is_prob', False):
                leader_prob_1 = float(getattr(leader_policy, '_fixed_action_prob'))
            else:
                leader_prob_1 = float(int(getattr(leader_policy, '_fixed_action')))
            leader_prob_0 = 1.0 - leader_prob_1
            for s in range(self.num_states):
                for a_l in range(self.num_leader_actions):
                    for a_f in range(self.num_actions):
                        next_state = self.transition_fn(s, a_l, a_f)
                        reward = self.reward_fn(s, a_l, a_f)
                        next_q = (
                            leader_prob_0 * Q_prev[next_state, 0]
                            + leader_prob_1 * Q_prev[next_state, 1]
                        )
                        next_val = self.soft_value_operator(next_q).item()
                        Q_new[s, a_l, a_f] = reward + self._discount * next_val
            max_diff = torch.max(torch.abs(Q_new - Q_prev)).item()
            self.Q = Q_new
        return max_diff, {}

@wrap_experiment
def main(ctxt, cfg):
    """Run a single training experiment with a fixed leader policy."""
    output_config_path = os.path.join(ctxt.snapshot_dir, 'config.yaml')
    OmegaConf.save(config=cfg, f=output_config_path)

    # Convert OmegaConf to regular Python objects to avoid type issues with torch
    l_params = OmegaConf.to_container(cfg['leader'], resolve=True)
    f_params = OmegaConf.to_container(cfg['follower'], resolve=True)

    set_seed(cfg['seed'])
    custom_set_seed(get_seed())

    trainer = Trainer(ctxt)

    env = GymEnv(env=cfg['env']['id'])
    env.seed(get_seed())
    eval_env = trainer.get_env_copy(env)
    eval_env.seed(get_seed())

    follower_algo = FixedLeaderSoftQIteration if f_params['algo'] == 'SoftQIteration' else get_algo(f_params['algo'])
    follower_kwargs_algo = SoftQIteration if f_params['algo'] == 'SoftQIteration' else follower_algo
    follower = FollowerWrapper(
        algo=follower_algo,
        fixed_policy=f_params['fixed_policy'],
        env=env,
        **kwargs_from_cfg(f_params, follower_kwargs_algo)
    )
    if f_params['algo'] == 'SoftQIteration':
        follower.algo_name = 'SoftQIteration'

    fixed_action = l_params.get('fixed_action', 1)
    leader_policy = FixedActionCategoricalPolicy(
        env_spec=env.spec.leader_policy_env_spec,
        fixed_action=fixed_action,
        hidden_sizes=l_params['policy_hidden_sizes'][0],
        hidden_nonlinearity=nn.ReLU,
        output_nonlinearity=None,
    )
    leader_qf = DiscreteMLPQFunction(
        env_spec=env.spec.leader_qf_env_spec,
        hidden_sizes=l_params['qf_hidden_sizes'][0],
        hidden_nonlinearity=nn.ReLU,
    )
    leader_replay_buffer = GammaReplayBuffer(
        env_spec=env.spec.leader_policy_env_spec,
        size=int(l_params['replay_buffer_size']),
        gamma=l_params['discount'],
    )
    leader_algo = get_algo(l_params['algo'])
    leader = leader_algo(
        env_spec=env.spec,
        policy=leader_policy,
        qf=leader_qf,
        replay_buffer=leader_replay_buffer,
        **kwargs_from_cfg(l_params, leader_algo)
    )
    leader.fixed_policy = True
    leader.is_deterministic_policy = False

    joint_policy = JointPolicy(
        env_spec=env.spec,
        leader_policy=leader.policy,
        follower_policy=follower.policy,
    )
    sampler = LocalSampler(
        agents=joint_policy,
        envs=env,
        max_episode_length=env.spec.max_episode_length,
        n_workers=1,
        worker_class=VecWorker,
        worker_args=dict(
            n_envs=l_params['sample_num_eps'],
        ),
    )
    eval_sampler = LocalSampler(
        agents=joint_policy,
        envs=eval_env,
        max_episode_length=eval_env.spec.max_episode_length,
        n_workers=1,
        worker_class=VecWorker,
        worker_args=dict(
            n_envs=l_params['eval_num_eps'],
        ),
    )

    follower.to()
    leader.to()

    trainer.setup(
        env=env,
        leader=leader,
        follower=follower,
        sampler=sampler,
        eval_env=eval_env,
        eval_sampler=eval_sampler,
        learner='leader',
    )
    trainer.train(
        n_epochs=l_params['n_epochs'],
        batch_size=int(l_params['sample_num_eps'] * env.spec.max_episode_length),
        max_total_env_steps=l_params['max_total_env_steps'],
    )

    return ctxt.snapshot_dir

In [ ]:
# Run grid search
all_results = []
all_log_dirs = []
start_time = time.time()
exp_count = 0

for fixed_action in grid_values:
    print(f"\n{'='*60}")
    print(f'Testing fixed_action = {fixed_action}')
    print(f"{'='*60}")

    for seed in SEEDS:
        exp_count += 1
        print(f'\n  Experiment {exp_count}: fixed_action={fixed_action}, seed={seed}')

        exp_config = OmegaConf.create(OmegaConf.to_container(config, resolve=True))
        exp_config['leader']['fixed_action'] = float(fixed_action)
        exp_config['seed'] = int(seed)

        exp_config['name'] = "grid_search"
        exp_config['sweep'] = {'env': None, 'leader': ['fixed_action'], 'follower': None}
        exp_config['leader']['n_epochs'] = 1
        exp_config['leader']['max_total_env_steps'] = 450
        exp_config['leader']['eval_num_eps'] = 20
        exp_config['leader']['init_steps'] = -1
        exp_config['leader']['fixed_policy'] = True

        ctxt, params = args_for_experiments(exp_config)
        ctxt['log_dir'] = f'{OUTPUT_DIR}/{ctxt["name"]}'
        log_dir = main(ctxt, cfg=params)
        all_log_dirs.append((fixed_action, seed, log_dir))
        print(f'  Results saved to: {log_dir}')

In [ ]:
print(f'Environment: {config["env"]["id"]}')

# Extract and aggregate results
print(f"\n{'='*60}")
print('Extracting results from experiment logs...')
print(f"{'='*60}\n")

for fixed_action, seed, log_dir in all_log_dirs:
    eval_progress_path = os.path.join(log_dir, 'eval', 'progress.csv')
    if os.path.exists(eval_progress_path):
        df = pd.read_csv(eval_progress_path)
        if 'Evaluation/AverageDiscountedTargetReturn' in df.columns:
            avg_target_return = df['Evaluation/AverageDiscountedTargetReturn'].iloc[0]
            all_results.append({
                'fixed_action': fixed_action,
                'seed': seed,
                'AverageDiscountedTargetReturn': avg_target_return,
            })
            print(f'fixed_action={fixed_action:3.1f}, seed={seed}: AverageDiscountedTargetReturn={avg_target_return:.2f}')
        else:
            print(f"WARNING: 'Evaluation/AverageDiscountedTargetReturn' not found in {eval_progress_path}")
    else:
        print(f'WARNING: Progress file not found: {eval_progress_path}')

results_df = pd.DataFrame(all_results)
if results_df.empty:
    raise RuntimeError('No results were collected. Check experiment logs.')

aggregated_df = results_df.groupby('fixed_action').agg({
    'AverageDiscountedTargetReturn': ['mean', 'std', 'count']
}).reset_index()
aggregated_df.columns = ['fixed_action', 'mean_return', 'std_return', 'num_seeds']
aggregated_df = aggregated_df.sort_values('fixed_action').reset_index(drop=True)

print(f"\n{'='*60}")
print('Grid Search Summary')
print(f"{'='*60}")
print(aggregated_df.to_string(index=False))

best_idx = aggregated_df['mean_return'].idxmax()
best_fixed_action = aggregated_df.loc[best_idx, 'fixed_action']
best_mean_return = aggregated_df.loc[best_idx, 'mean_return']
best_std_return = aggregated_df.loc[best_idx, 'std_return']

print(f"\n{'='*60}")
print(f'OPTIMAL FIXED_ACTION: {best_fixed_action}')
print(f'Mean AverageDiscountedTargetReturn: {best_mean_return:.4f} ± {best_std_return:.4f}')
print(f"{'='*60}\n")

os.makedirs(OUTPUT_DIR, exist_ok=True)
detailed_results_path = os.path.join(OUTPUT_DIR, 'grid_search_detailed.csv')
summary_results_path = os.path.join(OUTPUT_DIR, 'grid_search_summary.csv')
results_df.to_csv(detailed_results_path, index=False)
aggregated_df.to_csv(summary_results_path, index=False)

end_time = time.time()
elapsed_time = end_time - start_time
print(f'Detailed results saved to: {detailed_results_path}')
print(f'Summary results saved to: {summary_results_path}')
print(f'\nTotal experiments: {exp_count}')
print(f'Total time: {elapsed_time:.2f} seconds')